# Project · CineMate — a tool-calling movie chatbot (Groq + TMDB)

The LLM chats, decides when to call tools (`update_user_profile`, `search_movies`, `discover_movies`, `get_movie_details`), and answers with real data from **TMDB**. The agent loop is the same one you built in Module 8.

> 🔐 **Keys** — nothing in this notebook contains a key. In Colab open the **🔑 Secrets** panel (left sidebar), click **+ Add new secret**,
> type the *name* exactly as below, paste the *value*, and switch on **Notebook access**. If you skip this, you'll be asked once (hidden input).

| Secret name | What it is | Get it at |
|---|---|---|
| `GROQ_API_KEY` | Groq API key (LLM for CineMate) | https://console.groq.com/keys |
| `TMDB_API_KEY` | TMDB **v4 Read Access Token** (long `eyJ…` string, not the short v3 key) | https://www.themoviedb.org/settings/api |

**Never** paste a key into a code cell, commit it to GitHub, or share a notebook that shows it in an output. If one leaks, delete it in the provider's console and create a new one.

In [ ]:
!pip install -q groq requests python-dotenv

In [ ]:
import os, getpass

def get_secret(name, *, required=True, hint=""):
    """1) environment  2) Colab Secrets (key icon in the left sidebar)  3) hidden prompt.
    The value is kept in memory / os.environ only - it is never printed or saved in the notebook."""
    val = os.getenv(name)
    if not val:
        try:
            from google.colab import userdata          # works only inside Colab
            val = userdata.get(name)
        except Exception:
            val = None
    if not val and required:
        val = getpass.getpass(f"Paste {name} {hint}(input is hidden): ").strip()
    if val:
        os.environ[name] = val
    return val

## 1 · Files
`tools.py` holds the Python functions; `main.py` holds the system prompt, the tool schemas and the loop. (This is the original command-line version; the Streamlit version in the downloads folder adds a UI.)

In [ ]:
%cd /content
!mkdir -p cinemate
%cd cinemate

In [ ]:
%%writefile tools.py
import os
import requests

from typing import Optional
from dotenv import load_dotenv


# =========================================================
# CONFIGURATION
# =========================================================

load_dotenv()

TMDB_API_KEY = os.getenv("TMDB_API_KEY")

TMDB_BASE_URL = "https://api.themoviedb.org/3"

TMDB_HEADERS = {
    "Authorization": f"Bearer {TMDB_API_KEY}",
    "accept": "application/json",
}


# =========================================================
# IN-MEMORY USER PROFILE
# =========================================================

user_profile = {
    "name": None,

    "favorite_movies": [],

    "favorite_genres": [],

    "disliked_genres": [],

    "preferred_languages": [],

    "favorite_actors": [],

    "favorite_directors": [],

    "watched_movies": [],

    "likes_hidden_gems": None,
}


# =========================================================
# HELPER
# =========================================================

def add_unique(existing_list, new_items):
    """
    Add items to a list without duplicates.
    """

    if not new_items:
        return existing_list

    for item in new_items:

        if item and item not in existing_list:
            existing_list.append(item)

    return existing_list


def tmdb_request(endpoint, params=None):
    """
    Make a request to TMDB.
    """

    if not TMDB_API_KEY:
        raise ValueError(
            "TMDB_API_KEY is missing from .env"
        )

    url = f"{TMDB_BASE_URL}{endpoint}"

    response = requests.get(
        url,
        headers=TMDB_HEADERS,
        params=params or {},
        timeout=15,
    )

    response.raise_for_status()

    return response.json()


# =========================================================
# USER PROFILE TOOLS
# =========================================================

def update_user_profile(
    name: Optional[str] = None,
    favorite_movies: Optional[list[str]] = None,
    favorite_genres: Optional[list[str]] = None,
    disliked_genres: Optional[list[str]] = None,
    preferred_languages: Optional[list[str]] = None,
    favorite_actors: Optional[list[str]] = None,
    favorite_directors: Optional[list[str]] = None,
    watched_movies: Optional[list[str]] = None,
    likes_hidden_gems: Optional[bool] = None,
):
    """
    Update the user's movie preference profile.
    """

    if name is not None:
        user_profile["name"] = name

    if favorite_movies:
        add_unique(
            user_profile["favorite_movies"],
            favorite_movies
        )

    if favorite_genres:
        add_unique(
            user_profile["favorite_genres"],
            favorite_genres
        )

    if disliked_genres:
        add_unique(
            user_profile["disliked_genres"],
            disliked_genres
        )

    if preferred_languages:
        add_unique(
            user_profile["preferred_languages"],
            preferred_languages
        )

    if favorite_actors:
        add_unique(
            user_profile["favorite_actors"],
            favorite_actors
        )

    if favorite_directors:
        add_unique(
            user_profile["favorite_directors"],
            favorite_directors
        )

    if watched_movies:
        add_unique(
            user_profile["watched_movies"],
            watched_movies
        )

    if likes_hidden_gems is not None:
        user_profile["likes_hidden_gems"] = likes_hidden_gems

    return {
        "success": True,
        "profile": user_profile,
    }


def get_user_profile():
    """
    Return the current user profile.
    """

    return user_profile


# =========================================================
# TMDB TOOL 1
# SEARCH MOVIES
# =========================================================

def search_movies(query: str):
    """
    Search TMDB for movies by title or keyword.
    """

    data = tmdb_request(
        "/search/movie",
        {
            "query": query,
            "language": "en-US",
            "include_adult": False,
            "page": 1,
        }
    )

    results = []

    for movie in data.get("results", [])[:8]:

        results.append({
            "id": movie.get("id"),
            "title": movie.get("title"),
            "original_title": movie.get("original_title"),
            "release_date": movie.get("release_date"),
            "overview": movie.get("overview"),
            "rating": movie.get("vote_average"),
            "vote_count": movie.get("vote_count"),
            "popularity": movie.get("popularity"),
            "poster_path": movie.get("poster_path"),
            "original_language": movie.get(
                "original_language"
            ),
        })

    return {
        "query": query,
        "results": results,
    }


# =========================================================
# TMDB TOOL 2
# GET MOVIE GENRES
# =========================================================

def get_movie_genres():
    """
    Get TMDB movie genre names and IDs.
    """

    data = tmdb_request(
        "/genre/movie/list",
        {
            "language": "en",
        }
    )

    genres = {}

    for genre in data.get("genres", []):

        genres[genre["name"]] = genre["id"]

    return genres


# =========================================================
# TMDB TOOL 3
# DISCOVER MOVIES
# =========================================================

def discover_movies(
    genre_ids: Optional[list[int]] = None,
    year: Optional[int] = None,
    min_rating: Optional[float] = None,
    language: Optional[str] = None,
    sort_by: str = "popularity.desc",
):
    """
    Discover movies using TMDB filters.
    """

    params = {
        "language": "en-US",
        "sort_by": sort_by,
        "include_adult": False,
        "include_video": False,
        "page": 1,
    }

    if genre_ids:
        params["with_genres"] = ",".join(
            str(genre_id)
            for genre_id in genre_ids
        )

    if year:
        params["primary_release_year"] = year

    if min_rating is not None:
        params["vote_average.gte"] = min_rating

    if language:
        params["with_original_language"] = language

    data = tmdb_request(
        "/discover/movie",
        params
    )

    results = []

    for movie in data.get("results", [])[:10]:

        results.append({
            "id": movie.get("id"),
            "title": movie.get("title"),
            "original_title": movie.get("original_title"),
            "release_date": movie.get("release_date"),
            "overview": movie.get("overview"),
            "rating": movie.get("vote_average"),
            "vote_count": movie.get("vote_count"),
            "popularity": movie.get("popularity"),
            "poster_path": movie.get("poster_path"),
            "original_language": movie.get(
                "original_language"
            ),
        })

    return {
        "results": results,
    }


# =========================================================
# TMDB TOOL 4
# GET MOVIE DETAILS
# =========================================================

def get_movie_details(movie_id: int):
    """
    Get detailed information about a movie.
    """

    data = tmdb_request(
        f"/movie/{movie_id}",
        {
            "language": "en-US",
            "append_to_response": "credits,videos",
        }
    )

    # -----------------------------------------------------
    # Extract cast
    # -----------------------------------------------------

    cast = []

    for person in data.get(
        "credits",
        {}
    ).get(
        "cast",
        []
    )[:10]:

        cast.append({
            "name": person.get("name"),
            "character": person.get("character"),
        })

    # -----------------------------------------------------
    # Extract directors
    # -----------------------------------------------------

    directors = []

    for person in data.get(
        "credits",
        {}
    ).get(
        "crew",
        []
    ):

        if person.get("job") == "Director":

            directors.append(
                person.get("name")
            )

    # -----------------------------------------------------
    # Extract genres
    # -----------------------------------------------------

    genres = [
        genre.get("name")
        for genre in data.get("genres", [])
    ]

    # -----------------------------------------------------
    # Return useful information
    # -----------------------------------------------------

    return {
        "id": data.get("id"),
        "title": data.get("title"),
        "original_title": data.get("original_title"),
        "tagline": data.get("tagline"),
        "overview": data.get("overview"),
        "release_date": data.get("release_date"),
        "runtime": data.get("runtime"),
        "rating": data.get("vote_average"),
        "vote_count": data.get("vote_count"),
        "genres": genres,
        "original_language": data.get(
            "original_language"
        ),
        "budget": data.get("budget"),
        "revenue": data.get("revenue"),
        "poster_path": data.get("poster_path"),
        "backdrop_path": data.get("backdrop_path"),
        "cast": cast,
        "directors": directors,
        "homepage": data.get("homepage"),
    }

In [ ]:
%%writefile main.py
import os
import json

from dotenv import load_dotenv
from groq import Groq

from tools import (
    update_user_profile,
    get_user_profile,
    search_movies,
    get_movie_genres,
    discover_movies,
    get_movie_details,
)


# =========================================================
# CONFIGURATION
# =========================================================

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError(
        "GROQ_API_KEY is missing from .env"
    )


# =========================================================
# GROQ CLIENT
# =========================================================

client = Groq(
    api_key=GROQ_API_KEY
)


# =========================================================
# MODEL
# =========================================================

MODEL = "openai/gpt-oss-20b"


# =========================================================
# SYSTEM PROMPT
# =========================================================

SYSTEM_PROMPT = """
You are CineBot, an intelligent and friendly personal
movie recommendation assistant.

Your job is to understand the user's movie taste and
recommend real movies that they are likely to enjoy.

You have access to:

1. User profile tools
2. TMDB movie search
3. TMDB movie discovery
4. TMDB movie details
5. TMDB genre information

=========================================================
USER PROFILE
=========================================================

Gradually learn the user's movie preferences.

Whenever the user reveals useful information about:

- Their name
- Movies they love
- Movies they have watched
- Genres they enjoy
- Genres they dislike
- Preferred languages
- Favorite actors
- Favorite directors
- Whether they like hidden gems

call update_user_profile.

Do NOT ask the user to explicitly tell you to remember
something.

If they say:

"I love Interstellar"

you should update their profile.

If they say:

"I don't like horror"

you should update their profile.

If they say:

"I've already seen Inception"

you should update watched_movies.

=========================================================
CONVERSATION
=========================================================

During onboarding:

- Ask ONE question at a time.
- Do not interrogate the user.
- Keep the conversation natural.
- Don't ask for information already known.
- Try to understand their taste through conversation.

=========================================================
MOVIE DATA
=========================================================

TMDB is the source of truth for movie information.

Whenever possible, use TMDB tools instead of relying
on your own knowledge for:

- Movie titles
- Release dates
- Ratings
- Movie descriptions
- Cast
- Directors
- Genres
- Movie discovery

Never invent movie information.

=========================================================
RECOMMENDATIONS
=========================================================

When the user asks for movie recommendations:

1. Check their user profile.
2. Understand their preferences.
3. Use TMDB to find candidate movies.
4. Avoid movies they have already watched.
5. Avoid genres they dislike.
6. Prefer movies matching their favorite genres,
   actors, directors, languages and preferences.
7. Explain WHY each movie matches their taste.

Try to recommend 3-5 movies.

=========================================================
TMDB TOOLS
=========================================================

Use search_movies when:

- The user mentions a specific movie.
- You need to verify a movie.
- You need to find a movie by title.

Use get_movie_details when:

- You need detailed information about a movie.
- You need cast/director information.
- You need more information about a movie returned
  by search_movies or discover_movies.

Use get_movie_genres when:

- You need TMDB genre IDs.

Use discover_movies when:

- The user asks for recommendations.
- The user describes a genre, year, rating, language,
  or other movie preference.

=========================================================
IMPORTANT
=========================================================

Do not recommend a movie simply because you personally
know it.

Use TMDB to obtain real candidates.

Then use the user's profile and reasoning to determine
which candidates are the best fit.

Keep answers conversational rather than returning raw JSON.
"""


# =========================================================
# GROQ TOOLS
# =========================================================

tools = [

    # -----------------------------------------------------
    # UPDATE USER PROFILE
    # -----------------------------------------------------

    {
        "type": "function",

        "function": {

            "name": "update_user_profile",

            "description": """
            Update the user's movie preference profile.

            Call this whenever the user reveals information
            about their movie preferences.
            """,

            "parameters": {

                "type": "object",

                "properties": {

                    "name": {
                        "type": "string",
                        "description": "User's name"
                    },

                    "favorite_movies": {
                        "type": "array",
                        "items": {
                            "type": "string"
                        },
                        "description": "Movies the user loves"
                    },

                    "favorite_genres": {
                        "type": "array",
                        "items": {
                            "type": "string"
                        },
                        "description": "Genres the user likes"
                    },

                    "disliked_genres": {
                        "type": "array",
                        "items": {
                            "type": "string"
                        },
                        "description": "Genres the user dislikes"
                    },

                    "preferred_languages": {
                        "type": "array",
                        "items": {
                            "type": "string"
                        },
                        "description": "Preferred movie languages"
                    },

                    "favorite_actors": {
                        "type": "array",
                        "items": {
                            "type": "string"
                        },
                        "description": "Favorite actors"
                    },

                    "favorite_directors": {
                        "type": "array",
                        "items": {
                            "type": "string"
                        },
                        "description": "Favorite directors"
                    },

                    "watched_movies": {
                        "type": "array",
                        "items": {
                            "type": "string"
                        },
                        "description": "Movies the user has watched"
                    },

                    "likes_hidden_gems": {
                        "type": "boolean",
                        "description": """
                        Whether the user likes underrated
                        or lesser-known movies
                        """
                    },
                },
            },
        },
    },

    # -----------------------------------------------------
    # GET USER PROFILE
    # -----------------------------------------------------

    {
        "type": "function",

        "function": {

            "name": "get_user_profile",

            "description": """
            Get the user's current movie preference profile.
            Use this before making personalized recommendations.
            """,

            "parameters": {
                "type": "object",
                "properties": {},
            },
        },
    },

    # -----------------------------------------------------
    # SEARCH MOVIES
    # -----------------------------------------------------

    {
        "type": "function",

        "function": {

            "name": "search_movies",

            "description": """
            Search TMDB for movies by title or keyword.

            Use this when the user mentions a specific movie
            or when you need to find a movie.
            """,

            "parameters": {

                "type": "object",

                "properties": {

                    "query": {
                        "type": "string",
                        "description": "Movie title or search query"
                    }

                },

                "required": [
                    "query"
                ],
            },
        },
    },

    # -----------------------------------------------------
    # GET MOVIE GENRES
    # -----------------------------------------------------

    {
        "type": "function",

        "function": {

            "name": "get_movie_genres",

            "description": """
            Get the list of movie genres and their TMDB IDs.

            Use this when you need genre IDs for discovering
            movies.
            """,

            "parameters": {
                "type": "object",
                "properties": {},
            },
        },
    },

    # -----------------------------------------------------
    # DISCOVER MOVIES
    # -----------------------------------------------------

    {
        "type": "function",

        "function": {

            "name": "discover_movies",

            "description": """
            Discover movies from TMDB using filters.

            Use this for movie recommendations.

            genre_ids should contain TMDB genre IDs.

            min_rating can be used when the user asks for
            highly rated movies.

            year can be used when the user specifies a year.

            language is a two-letter ISO language code such
            as 'en', 'hi', 'fr', etc.

            sort_by can be:
            popularity.desc
            vote_average.desc
            primary_release_date.desc
            """,

            "parameters": {

                "type": "object",

                "properties": {

                    "genre_ids": {
                        "type": "array",
                        "items": {
                            "type": "integer"
                        },
                        "description": "TMDB genre IDs"
                    },

                    "year": {
                        "type": "integer",
                        "description": "Release year"
                    },

                    "min_rating": {
                        "type": "number",
                        "description": "Minimum TMDB rating"
                    },

                    "language": {
                        "type": "string",
                        "description": "Original language ISO code"
                    },

                    "sort_by": {
                        "type": "string",
                        "description": """
                        TMDB sorting method.
                        Example: popularity.desc
                        """
                    },

                },
            },
        },
    },

    # -----------------------------------------------------
    # MOVIE DETAILS
    # -----------------------------------------------------

    {
        "type": "function",

        "function": {

            "name": "get_movie_details",

            "description": """
            Get detailed information about a movie from TMDB.

            Use this when you need detailed information,
            cast, director, genres, runtime or other metadata.
            """,

            "parameters": {

                "type": "object",

                "properties": {

                    "movie_id": {
                        "type": "integer",
                        "description": "TMDB movie ID"
                    }

                },

                "required": [
                    "movie_id"
                ],
            },
        },
    },
]


# =========================================================
# FUNCTION MAP
# =========================================================

available_functions = {

    "update_user_profile": update_user_profile,

    "get_user_profile": get_user_profile,

    "search_movies": search_movies,

    "get_movie_genres": get_movie_genres,

    "discover_movies": discover_movies,

    "get_movie_details": get_movie_details,
}


# =========================================================
# CHAT / AGENT LOOP
# =========================================================

def run_chat(messages):

    """
    Run the Groq agent.

    The agent can make multiple function calls before
    producing the final answer.
    """

    while True:

        response = client.chat.completions.create(

            model=MODEL,

            messages=messages,

            tools=tools,

            tool_choice="auto",

            temperature=0.7,
        )

        response_message = response.choices[0].message

        # Add assistant message to conversation
        messages.append(response_message)

        # -------------------------------------------------
        # NO TOOL CALL
        # -------------------------------------------------

        if not response_message.tool_calls:

            return response_message.content

        # -------------------------------------------------
        # TOOL CALLS
        # -------------------------------------------------

        for tool_call in response_message.tool_calls:

            function_name = tool_call.function.name

            print(
                f"\n  🔧 Tool: {function_name}"
            )

            try:

                function_args = json.loads(
                    tool_call.function.arguments
                )

                function_to_call = (
                    available_functions.get(
                        function_name
                    )
                )

                if function_to_call is None:

                    tool_result = {
                        "success": False,
                        "error": (
                            f"Unknown function: "
                            f"{function_name}"
                        ),
                    }

                else:

                    tool_result = function_to_call(
                        **function_args
                    )

                print(
                    "  ✓ Done"
                )

            except Exception as error:

                print(
                    f"  ✗ Tool error: {error}"
                )

                tool_result = {
                    "success": False,
                    "error": str(error),
                }

            # -------------------------------------------------
            # SEND TOOL RESULT BACK TO GROQ
            # -------------------------------------------------

            messages.append({

                "role": "tool",

                "tool_call_id": tool_call.id,

                "content": json.dumps(
                    tool_result
                ),
            })

        # -------------------------------------------------
        # LOOP AGAIN
        # -------------------------------------------------

        # Groq now receives the tool results and decides
        # whether it needs another tool or can answer.


# =========================================================
# MAIN CLI
# =========================================================

def main():

    print()
    print("=" * 65)

    print(
        "🎬 CineBot — Your Personal Movie Recommendation Assistant"
    )

    print("=" * 65)

    print()
    print(
        "I'll learn your movie taste as we chat."
    )

    print(
        "Type 'profile' to see what I know about you."
    )

    print(
        "Type 'exit' to quit."
    )

    print()


    # -----------------------------------------------------
    # CONVERSATION MEMORY
    # -----------------------------------------------------

    messages = [

        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        }

    ]


    # -----------------------------------------------------
    # INITIAL GREETING
    # -----------------------------------------------------

    print(
        "🤖 CineBot: "
        "Hey! 👋 Let's get to know your movie taste."
    )

    print()

    print(
        "What's one movie you absolutely loved?"
    )

    print()


    # -----------------------------------------------------
    # CHAT LOOP
    # -----------------------------------------------------

    while True:

        try:

            user_input = input("You: ").strip()

        except (KeyboardInterrupt, EOFError):

            print(
                "\n\n👋 Goodbye!\n"
            )

            break


        if not user_input:
            continue


        # -------------------------------------------------
        # EXIT
        # -------------------------------------------------

        if user_input.lower() in {
            "exit",
            "quit",
            "bye",
        }:

            print()
            print(
                "🤖 CineBot: "
                "Thanks for chatting! See you next time. 🎬"
            )
            print()

            break


        # -------------------------------------------------
        # PROFILE COMMAND
        # -------------------------------------------------

        if user_input.lower() == "profile":

            profile = get_user_profile()

            print()
            print("🧠 YOUR MOVIE PROFILE")
            print("-" * 40)

            print(
                json.dumps(
                    profile,
                    indent=2,
                    ensure_ascii=False,
                )
            )

            print()

            continue


        # -------------------------------------------------
        # ADD USER MESSAGE
        # -------------------------------------------------

        messages.append({

            "role": "user",

            "content": user_input,

        })


        # -------------------------------------------------
        # CALL AGENT
        # -------------------------------------------------

        try:

            print()

            response = run_chat(
                messages
            )

            print(
                f"🤖 CineBot: {response}"
            )

            print()

        except Exception as error:

            print()
            print(
                f"❌ Error: {error}"
            )
            print()


# =========================================================
# ENTRY POINT
# =========================================================

if __name__ == "__main__":

    main()

## 2 · Keys
The TMDB value must be the **v4 “API Read Access Token”** (the long one), because it is sent as `Authorization: Bearer …`.

In [ ]:
import os, getpass

def get_secret(name, *, required=True, hint=""):
    """1) environment  2) Colab Secrets (key icon in the left sidebar)  3) hidden prompt.
    The value is kept in memory / os.environ only - it is never printed or saved in the notebook."""
    val = os.getenv(name)
    if not val:
        try:
            from google.colab import userdata          # works only inside Colab
            val = userdata.get(name)
        except Exception:
            val = None
    if not val and required:
        val = getpass.getpass(f"Paste {name} {hint}(input is hidden): ").strip()
    if val:
        os.environ[name] = val
    return val

get_secret("GROQ_API_KEY", hint="(console.groq.com/keys) ")
get_secret("TMDB_API_KEY", hint="(TMDB v4 Read Access Token) ")

## 3 · Run it
`main.py` reads from `input()`; `%run` lets Colab show an input box: type a message, press Enter, type `exit` to stop. Try: *“I love Interstellar and hate horror. Suggest 3 movies.”*

In [ ]:
%run main.py